In [ ]:
import re, sqlite3, unicodedata
import pandas as pd
from collections import Counter

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 300)

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("""
    SELECT Patient_ID, Study_ID, visit_datetime, karte_text
    FROM karte
    WHERE karte_text LIKE '%浮腫%' OR karte_text LIKE '%むくみ%'
       OR karte_text LIKE '%ムクミ%' OR karte_text LIKE '%edema%'
""", conn)
conn.close()
print(f"浮腫系の語を含むカルテ: {len(df):,} 件 / {df['Patient_ID'].nunique():,} 人")

KW = re.compile(r"(?:浮腫|むくみ|ムクミ|EDEMA)")
PUNCT = r"[、。\n]"

after_c, before_c, rows = Counter(), Counter(), []
for pid, vd, raw in zip(df["Patient_ID"], df["visit_datetime"], df["karte_text"]):
    t = unicodedata.normalize("NFKC", str(raw)).upper()
    for m in KW.finditer(t):
        after  = re.split(PUNCT, t[m.end(): m.end() + 12])[0]      # 直後（句読点まで）
        before = re.split(PUNCT, t[max(0, m.start() - 12): m.start()])[-1]  # 直前
        after_c[after] += 1
        before_c[before] += 1
        rows.append((pid, vd, before + "【" + m.group(0) + "】" + after))

ctx = pd.DataFrame(rows, columns=["Patient_ID", "visit_datetime", "context"])
print(f"出現箇所: {len(ctx):,}")

In [ ]:
pd.DataFrame(after_c.most_common(150), columns=["浮腫の直後", "件数"])

In [ ]:
pd.DataFrame(before_c.most_common(150), columns=["浮腫の直前", "件数"])

In [ ]:
cands = ["増悪", "悪化", "出現", "増強", "増加", "進行", "著明", "目立", "強く", "強い",
         "ひどい", "増えて", "増えた", "拡大", "広が", "新規", "新た", "出て", "認める", "あり"]
summary = {w: ctx["context"].str.contains(w, regex=False).sum() for w in cands}
print(pd.Series(summary).sort_values(ascending=False))

# 語ごとの実例を見る
w = "増悪"
ctx[ctx["context"].str.contains(w, regex=False)].head(30)

これより下は，修正コードになる．

In [ ]:
GAP = r"[^、。,;；\r\n]{0,10}"
WORSEN = (r"(?:出現|増悪|悪化|増強|増加|↑|強まっ|強くな|ひどく|増え|進行"
          r"|拡大|広が|目立ってき|出てき|出た|新規)")
CONDITION_REGEX = re.compile(
    r"(時|場合|あれば|ようなら|際|連絡|ください|下さい|有無|確認|チェック"
    r"|リスク|注意|既往|疑い|可能性|恐れ|おそれ)")
NEGATION_REGEX = re.compile(r"(なし|無し|なく|無く|認めず|認めない|否定|ない|\(\s*-\s*\))")